<a href="https://colab.research.google.com/github/jcarlosfelix/jupyter_notebook/blob/main/Taller_Practico_Preprocesamiento_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TecNM / Instituto Tecnológico de Tijuana
## Materia: Analítica de Datos (IAD-2403)
### Profesor: José Carlos Félix Saúl
---
# Taller Práctico: Preprocesamiento de Datos - Muestreo y Cuantificación

**Objetivos de la sesión:**
1. Importar y explorar datasets estructurados usando `pandas`.
2. Aplicar filtrados condicionales.
3. Realizar limpieza de información anómala o nula mediante funciones `lambda` y `.apply()`.
4. Cuantificar datos categóricos (Label Encoding, Ordinal Encoding y One-Hot Encoding).
5. Experimentar con distintas técnicas de muestreo en Pandas (Aleatorio Simple, Estratificado y Sistemático).
6. Generar visualizaciones descriptivas para comparar muestras contra la población total.

### [Código 1: Importación de Librerías]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Configuración gráfica para el taller
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

print("Librerías cargadas exitosamente.")

## Parte 1: Carga de Datos y Exploración Inicial
Cargamos el archivo CSV del taller y realizamos una inspección de la estructura, tipos de datos y primeros registros.

### [Código 2: Carga de Dataset]

In [ ]:
# Carga del dataset CSV del concentrado de información de la clase
archivo_csv = 'Actividad_Taller-Concentrado_Info.csv'

# Intentamos cargar el archivo especificando codificación estándar
try:

    #df = pd.read_csv(archivo_csv)
    df = pd.read_csv(archivo_csv, encoding='latin1')

    print(f"Dataset cargado correctamente. Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas.")
except FileNotFoundError:
    print("Asegúrate de que el archivo CSV se encuentra en la misma carpeta que este Notebook.")

# Muestra general
df.head()

### [Código 3: Inspección Estructural]

In [ ]:
# Verificación de tipos de datos y valores nulos
print("--- INFORMACIÓN DEL DATAFRAME ---")
df.info()

print("\n--- RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ---")
display(df.describe())

print("\n--- PRIMERAS CATEGORÍAS EN VARIABLES DE TEXTO ---")
display(df.describe(include=['O']))

## Parte 2: Filtrado por Condición
El filtrado por condición permite aislar subconjuntos de interés para análisis específicos (ej. respuestas por rango de edad o ubicación geográfica).

### [Código 4: Ejercicios de Filtrado]

### Identificación de columnas de texto y numéricas para adaptar las consultas automáticamente

In [ ]:
# Identificación de columnas de texto y numéricas para adaptar las consultas automáticamente
col_num = df.select_dtypes(include=[np.number]).columns.tolist()
col_cat = df.select_dtypes(include=['object']).columns.tolist()

### Ejercicio 1: Filtrado numérico (ej. valores superiores al promedio en la primera columna numérica)

In [ ]:
# Ejercicio 1: Filtrado numérico (ej. valores superiores al promedio en la primera columna numérica)
if col_num:
    col_ref_num = col_num[0]
    promedio_val = df[col_ref_num].mean()

    filtro_num = df[df[col_ref_num] > promedio_val]

    print(f"Registros con '{col_ref_num}' superior al promedio ({promedio_val:.2f}): {len(filtro_num)}")
    display(filtro_num.head(3))

### Ejercicio 2: Filtrado por coincidencia categórica

In [ ]:
# Ejercicio 2: Filtrado por coincidencia categórica
if col_cat:
    col_ref_cat = col_cat[0]
    categoria_frecuente = df[col_ref_cat].mode()[0]

    filtro_cat = df[df[col_ref_cat] == categoria_frecuente]

    print(f"\nRegistros filtrados por '{col_ref_cat}' == '{categoria_frecuente}': {len(filtro_cat)}")
    display(filtro_cat.head(3))

## Parte 3: Limpieza de Datos y Uso de Funciones Lambda
Siguiendo las metodologías del taller, procesaremos valores faltantes, registros fuera de rango (como valores negativos o nulos) y estandarizaremos textos.

### [Código 5: Limpieza y Transformación]

In [ ]:
# Copia de trabajo para la limpieza
df_clean = df.copy()

# 1. Estandarización de texto (eliminar espacios en blanco sobrantes y convertir a formato Titulo)
for col in col_cat:
    df_clean[col] = df_clean[col].apply(lambda x: x.strip().title() if isinstance(x, str) else x)

# 2. Corrección/Imputación de valores numéricos incoherentes o nulos mediante lambda[cite: 1]
for col in col_num:
    # Calculamos la media de valores válidos (mayores a cero)[cite: 1]
    media_valida = df_clean[df_clean[col] > 0][col].mean()
    if np.isnan(media_valida):
        media_valida = 0

    # Aplicación de lambda para reemplazar nulos o negativos por la media[cite: 1]
    df_clean[col] = df_clean[col].apply(
        lambda x: media_valida if pd.isnull(x) or (isinstance(x, (int, float)) and x < 0) else x
    )

print("Estado de nulos tras la limpieza:")
print(df_clean.isnull().sum())

## Parte 4: Cuantificación de Datos
Convertiremos las variables de texto/categóricas a formatos numéricos aptos para modelos de cómputo y aprendizaje automático:
* **Label Encoding / Mapeo Diccionario:** Asignación de un entero a cada categoría.
* **One-Hot Encoding:** Creación de columnas binarias ($0$ o $1$) por cada categoría.

### [Código 6: Técnicas de Cuantificación]

### Demostración de Label Encoding en columna

In [ ]:
df_cuant = df_clean.copy()

if len(col_cat) > 0:
    # 1. Uso de Map con Diccionario (Label Encoding manual)
    col_target1 = col_cat[0]

    categorias_unicas = df_cuant[col_target1].unique()
    print(categorias_unicas)


In [ ]:
    mapa_categorias = {cat: idx + 1 for idx, cat in enumerate(categorias_unicas)}
    print(mapa_categorias)

    df_cuant[f'{col_target1}_Cod_Map'] = df_cuant[col_target1].map(mapa_categorias)

    # 2. LabelEncoder de Scikit-Learn
    le = LabelEncoder()
    df_cuant[f'{col_target1}_Cod_LE'] = le.fit_transform(df_cuant[col_target1])

    print(f"Demostración de Label Encoding en columna '{col_target1}':")
    display(df_cuant[[col_target1, f'{col_target1}_Cod_Map', f'{col_target1}_Cod_LE']].head())

### Demostración de One-Hot Encoding en columna

In [ ]:
if len(col_cat) > 1:
    # 3. One-Hot Encoding con Pandas[cite: 1]
    col_target2 = col_cat[1]
    df_onehot = pd.get_dummies(df_cuant[col_target2], prefix=col_target2, dtype=int)

    print(f"\nDemostración de One-Hot Encoding en columna '{col_target2}':")
    display(pd.concat([df_cuant[[col_target2]], df_onehot], axis=1).head())

## Parte 5: Métodos de Muestreo en Pandas
Evaluaremos tres estrategias para seleccionar muestras representativas de la población total:
1. **Muestreo Aleatorio Simple:** Selección sin criterio específico (`df.sample`).
2. **Muestreo Estratificado:** Muestreo proporcional por grupos/estratos.
3. **Muestreo Sistemático:** Selección periódica cada $k$-ésimo elemento.

### [Código 7: Ejecución de Muestreos]

In [ ]:
tamano_población = len(df_clean)
tamano_muestra = max(3, int(tamano_población * 0.3)) # Tomamos ~30% como muestra

print(f"Población total: {tamano_población} registros.")

### 1. Muestreo Aleatorio Simple

In [ ]:
# 1. Muestreo Aleatorio Simple
muestra_simple = df_clean.sample(n=tamano_muestra, random_state=42)

print(f"Muestra Aleatoria Simple: {len(muestra_simple)} registros.")
print(muestra_simple)

### 2. Muestreo Sistemático (tomando cada k-ésimo elemento)

In [ ]:
# 2. Muestreo Sistemático (tomando cada k-ésimo elemento)
k = max(1, tamano_población // tamano_muestra)
muestra_sistemática = df_clean.iloc[::k].head(tamano_muestra)

print(f"Muestra Sistemática (k={k}): {len(muestra_sistemática)} registros.")
print(muestra_sistemática)

### 3. Muestreo Estratificado (tomando muestras según una variable categórica)

In [ ]:
# 3. Muestreo Estratificado (tomando muestras según una variable categórica)
if len(col_cat) > 0:
    estrato_col = col_cat[0]
    muestra_estratificada = df_clean.groupby(estrato_col, group_keys=False).apply(
        lambda x: x.sample(frac=0.3, random_state=42) if len(x) > 1 else x
    )
else:
    muestra_estratificada = muestra_simple

print(f"Muestra Estratificada: {len(muestra_estratificada)} registros.")
#print(muestra_estratificada)

## Parte 6: Visualización de Resultados
Comparamos las distribuciones de la **Población Total** con las de las **Muestras** para analizar la representatividad.

In [ ]:
if len(col_cat) > 0:
    col_vis = col_cat[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

    # Gráfico 1: Población Total
    df_clean[col_vis].value_counts(normalize=True).plot(
        kind='bar', ax=axes[0], color='skyblue', edgecolor='black'
    )
    axes[0].set_title(f"Distribución Poblacional - '{col_vis}'")
    axes[0].set_ylabel("Proporción")
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

    # Gráfico 2: Muestra Estratificada vs Aleatoria
    muestra_simple[col_vis].value_counts(normalize=True).plot(
        kind='bar', ax=axes[1], color='coral', edgecolor='black'
    )
    axes[1].set_title(f"Distribución Muestra Aleatoria Simple - '{col_vis}'")
    axes[1].set_ylabel("Proporción")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
# Histogramas si existen variables numéricas
if len(col_num) > 0:
    col_num_vis = col_num[0]
    plt.figure(figsize=(8, 4))
    sns.kdeplot(df_clean[col_num_vis], label='Población', fill=True, alpha=0.3)
    sns.kdeplot(muestra_simple[col_num_vis], label='Muestra Aleatoria', fill=True, alpha=0.3)
    plt.title(f"Comparación de Densidad en '{col_num_vis}'")
    plt.xlabel(col_num_vis)
    plt.ylabel("Densidad")
    plt.legend()
    plt.show()

## Parte 7: Preguntas de Reflexión (Para Discusión en Clase)

1. **¿Cuándo es preferible utilizar un muestreo estratificado sobre uno aleatorio simple?**
2. **¿Qué implicaciones o sesgos genera realizar una cuantificación por ordinales (Label Encoding) sobre variables nominales sin orden inherente?**
3. **¿Cómo influyó el tamaño de la muestra en la similitud gráfica entre la muestra y la población total?**
4. **Analiza el caso del error de muestreo:** Si encuestáramos únicamente a un subgrupo con cierta característica particular, ¿por qué fallaría la predicción hacia los demás subgrupos?